# Phase 3 — Embedding & Hybrid Index Analysis

Goal: embed the real AAPL 10-K 2023 chunk corpus (737 chunks, produced by Phase 2's `scripts/process_filings.py`) with BGE-M3, fit BM25 over the same corpus, inspect what both representations actually look like, then run a dense-vs-sparse retrieval comparison against the live Qdrant collection built by `scripts/build_index.py` — to sanity-check Phase 3 before Phase 4/5 build on top of it.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import json

from qdrant_client import QdrantClient

from src.indexing.embeddings import BGEEmbedder
from src.indexing.sparse_indexer import BM25SparseEncoder
from src.processing.models import Chunk
from src.utils.config import load_config

cfg = load_config()
chunks_path = Path("../data/processed/AAPL/10-K_2023/chunks.json")
chunks = [Chunk.model_validate(c) for c in json.loads(chunks_path.read_text())]
print(f"{len(chunks)} chunks loaded from {chunks_path}")

737 chunks loaded from ../data/processed/AAPL/10-K_2023/chunks.json


## 1. Dense embeddings (BGE-M3)

Embed a handful of chunks directly (not the full 737 — `scripts/build_index.py` already did that for the real index) to inspect vector shape, normalization, and how cosine similarity behaves on two chunks we know are topically related vs. unrelated.

In [2]:
embedder = BGEEmbedder(cfg.embeddings.dense_model, batch_size=cfg.embeddings.batch_size)

sample = chunks[:8]
vecs = embedder.embed([c.text for c in sample])
print("shape:", vecs.shape)
print("norm of each vector (should be ~1.0, normalize_embeddings=True):")
print((vecs**2).sum(axis=1) ** 0.5)

[06/23/26 14:46:57] INFO     Loading dense embedding model: BAAI/bge-m3

                    INFO     Use pytorch device_name: mps

                    INFO     Load pretrained SentenceTransformer: BAAI/bge-m3

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

shape: (8, 1024)
norm of each vector (should be ~1.0, normalize_embeddings=True):
[1. 1. 1. 1. 1. 1. 1. 1.]


In [3]:
# cosine sim between every pair in the 8-chunk sample (vectors already unit-normalized,
# so dot product == cosine similarity)
sim = vecs @ vecs.T

for i in range(len(sample)):
    for j in range(i + 1, len(sample)):
        print(f"[{i}] vs [{j}]  sim={sim[i, j]:.3f}")
        print("   ", sample[i].text[:70].replace("\n", " "))
        print("   ", sample[j].text[:70].replace("\n", " "))

[0] vs [1]  sim=0.697
    Item 1. Business Company Background The Company designs, manufactures 
    . Mac Mac ® is the Company's line of personal computers based on its m
[0] vs [2]  sim=0.684
    Item 1. Business Company Background The Company designs, manufactures 
    . The Company's line of smartwatches, based on its watchOS ® operating
[0] vs [3]  sim=0.648
    Item 1. Business Company Background The Company designs, manufactures 
    . Accessories includes Apple-branded and third-party accessories. Appl
[0] vs [4]  sim=0.626
    Item 1. Business Company Background The Company designs, manufactures 
    . The offerings provide priority access to Apple technical support, ac
[0] vs [5]  sim=0.625
    Item 1. Business Company Background The Company designs, manufactures 
    . Digital Content The Company operates various platforms, including th
[0] vs [6]  sim=0.618
    Item 1. Business Company Background The Company designs, manufactures 
    . The Company also offers digital conte

## 2. Sparse vectors (BM25)

Fit BM25 over the **full** 737-chunk corpus (IDF only makes sense corpus-wide — see `docs/phases/phase3_indexing.md`), then inspect the sparse vector for one chunk: which terms get the most weight, and why.

In [4]:
bm25 = BM25SparseEncoder()
texts = [c.text for c in chunks]
bm25.fit(texts)
sparse_vecs = bm25.encode_all()

avg_nnz = sum(len(v.indices) for v in sparse_vecs) / len(sparse_vecs)
print(f"vocab size: {len(bm25._vocab)}")
print(f"avg sparse vector nnz (nonzero terms): {avg_nnz:.1f}")

[06/23/26 14:47:19] INFO     BM25 fit: 737 docs, vocab size 3206

vocab size: 3206
avg sparse vector nnz (nonzero terms): 39.0


In [5]:
# top-weighted terms for one chunk — high BM25 weight = rare-ish term that appears
# repeatedly in THIS chunk relative to the rest of the corpus
idx = 0
id_to_term = {v: k for k, v in bm25._vocab.items()}
vec = sparse_vecs[idx]
top = sorted(zip(vec.indices, vec.values), key=lambda t: -t[1])[:10]

print("chunk text:", chunks[idx].text[:200].replace("\n", " "))
print("\ntop BM25 terms:")
for term_id, weight in top:
    print(f"  {id_to_term[term_id]:<20s} {weight:.3f}")

chunk text: Item 1. Business Company Background The Company designs, manufactures and markets smartphones, personal computers, tablets, wearables and accessories, and sells a variety of related services. The Comp

top BM25 terms:
  iphone               7.551
  smartphones          6.357
  background           5.440
  manufactures         5.440
  line                 5.388
  se                   4.990
  ends                 4.694
  last                 4.694
  saturday             4.694
  ios                  4.694


## 3. Dense vs. sparse retrieval — live Qdrant collection

Query the real `financial_rag` collection built by `python scripts/build_index.py --ticker AAPL --years 2023` (embedded Qdrant at `data/index/qdrant_local/`). Compare top hits for a semantic-style question (dense should win) vs. a keyword-heavy lookup (sparse should win).

In [6]:
client = QdrantClient(path="../" + cfg.indexing.qdrant_path)
info = client.get_collection(cfg.indexing.collection_name)
print(f"collection '{cfg.indexing.collection_name}': {info.points_count} points")

collection 'financial_rag': 737 points


In [7]:
def show_hits(label, hits):
    print(f"--- {label} ---")
    for h in hits:
        print(f"  score={h.score:.3f}  {h.payload['text'][:90].replace(chr(10), ' ')}")
    print()


query = "What factors could materially affect Apple's future revenue?"
qvec = embedder.embed([query])[0]
dense_hits = client.query_points(
    cfg.indexing.collection_name, query=qvec.tolist(), using="dense", limit=3
).points
show_hits(f"DENSE: {query!r}", dense_hits)

sparse_query_vec = bm25.encode_query(query)
sparse_hits = client.query_points(
    cfg.indexing.collection_name, query=sparse_query_vec, using="sparse", limit=3
).points
show_hits(f"SPARSE: {query!r}", sparse_hits)

--- DENSE: "What factors could materially affect Apple's future revenue?" ---
  score=0.689  . The Company could also experience a significant increase in payment card transaction cos
  score=0.663  . Macroeconomic Conditions Macroeconomic conditions, including inflation, changes in inter
  score=0.654  . Future changes could also affect what the Company charges developers for access to its p

--- SPARSE: "What factors could materially affect Apple's future revenue?" ---
  score=70.207  . Future changes could also affect what the Company charges developers for access to its p
  score=63.075  . Future changes could also affect what the Company charges developers for access to its p
  score=41.746  . If disputes and conflicts further escalate in the future, actions by governments in resp



In [8]:
# keyword-heavy lookup — exact term match (a specific product name) should favor sparse
query = "iPhone 15 Pro"
qvec = embedder.embed([query])[0]
dense_hits = client.query_points(
    cfg.indexing.collection_name, query=qvec.tolist(), using="dense", limit=3
).points
show_hits(f"DENSE: {query!r}", dense_hits)

sparse_query_vec = bm25.encode_query(query)
sparse_hits = client.query_points(
    cfg.indexing.collection_name, query=sparse_query_vec, using="sparse", limit=3
).points
show_hits(f"SPARSE: {query!r}", sparse_hits)

--- DENSE: 'iPhone 15 Pro' ---
  score=0.648  Item 15
  score=0.599  . Third Quarter 2023: • MacBook Air 15", Mac Studio and Mac Pro; • Apple Vision ProTM, the
  score=0.529  . Apple Inc

--- SPARSE: 'iPhone 15 Pro' ---
  score=68.564  . Third Quarter 2023: • MacBook Air 15", Mac Studio and Mac Pro; • Apple Vision ProTM, the
  score=55.241  Item 1. Business Company Background The Company designs, manufactures and markets smartpho
  score=52.408  . (2) Services net sales include amortization of the deferred value of services bundled in



## Takeaways

- BGE-M3 vectors are unit-normalized (1024-dim) — cosine similarity reduces to a dot product, which is what `dense_indexer.py` configures the collection's `Distance.COSINE` metric to compute.
- BM25 sparse vectors are sparse for a reason: ~3,200 vocab terms corpus-wide, but each chunk only has a few dozen nonzero weights — this is what makes Qdrant's sparse HNSW index cheap to query.
- Note this notebook's `bm25.encode_query()` here uses a vocab freshly fit in this session (over the same 737 chunks), not the persisted `data/index/bm25_vocab.json` — Phase 5's retrieval code should load the persisted vocab instead of refitting, so query-time term ids stay consistent with whatever was actually indexed (see `BM25SparseEncoder.save_vocab` docstring).
- Both query types returned plausible top hits against the real index — confirms the Phase 3 pipeline (embed → BM25 fit/encode → upsert) produces a genuinely searchable hybrid index, not just one that loads without errors.